In [1]:
from unsloth import FastLanguageModel
from datasets import load_dataset, disable_caching
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
import torch
import pandas as pd

disable_caching()
torch.cuda.empty_cache()

[unsloth.import_fixes|WARNING]Unsloth: torch==2.10.0+cu130 requires torchvision>=0.25.0, but found torchvision==0.15.2a0. Try updating torchvision via `pip install --upgrade "torchvision>=0.25.0"`. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\maria\conda\envs\project1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\maria\conda\envs\project1\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: C:\Users\maria\conda\envs\project1\Lib\site-packages\torchvision\image.pyd'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
W0311 09:29:02.459000 24392 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\maria\conda\envs\project1\lib\site-packages\torchvision\datapoints\__

🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
MODEL_NAME = "unsloth/llama-2-7b-bnb-4bit"
DATA_PATH = "data/flipped_dataset.csv"
SEED = 42
MAX_SEQ_LENGTH = 256
SUBSET_SIZE = 5000 # 10K, 20K, 100K

In [3]:
dataset = load_dataset("csv", data_files=DATA_PATH)["train"]

# Shuffle and Subset
dataset = dataset.shuffle(seed=SEED)
dataset = dataset.select(range(min(SUBSET_SIZE, len(dataset))))

split = dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

In [4]:
def create_text_column(example):
    # Combine all candidate fields into a single string
    prompt = (
        f"Below is an instruction that describes a fair hiring decision task.\n\n"
        f"### Instruction:\n"
        f"You are an unbiased AI hiring assistant. \n"
        f"Make a decision based ONLY on qualifications. \n"
        f"Ignore gender and any demographic attributes.\n\n"
        f"Given the following candidate profile, determine whether the candidate should be hired.\n\n"
        f"Education Level: {example['education_level']}\n"
        f"Specialization Domain: {example['specialization_domain']}\n"
        f"Has Certification: {example['has_certification']}\n"
        f"Skill Count: {example['skill_count']}\n"
        f"Tech Skill Count: {example['tech_skill_count']}\n"
        f"Soft Skill Count: {example['soft_skill_count']}\n"
        f"Education Job Match: {example['education_job_match']}\n"
        f"Highest Qualification Level: {example['highest_qualification_level']}\n"
        f"Gender: {example['Gender']}\n\n"
        f"### Response:\n"
        f"{example['is_employed']}"
    )
    return {"text": prompt}

train_dataset = train_dataset.map(create_text_column)
eval_dataset = eval_dataset.map(create_text_column)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 4818.87 examples/s]


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map = "auto",
     max_memory = {
        0: "9GB",
        "cpu": "48GB"
    }
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 10.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [7]:
EOS_TOKEN = tokenizer.eos_token

def format_prompt(example):
    prompt = example["text"] + EOS_TOKEN
    tokenized = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LENGTH)
    tokenized["labels"] = tokenized["input_ids"].copy()  # supervised fine-tuning
    return tokenized

In [8]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    output_dir="outputs",
    run_name=f"llama2_bias_finetune_{SUBSET_SIZE}",
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    lr_scheduler_type="linear",
    seed=SEED,
    report_to="none"
)

In [9]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    formatting_func=format_prompt,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=10,
        early_stopping_threshold=0.001
    )]
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 1000/1000 [00:00<00:00, 5518.20 examples/s]


In [10]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 3 | Total steps = 3,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


Step,Training Loss,Validation Loss
50,0.149700,0.120096
100,0.052400,0.055809
150,0.056700,0.053991
200,0.053700,0.052598
250,0.054600,0.049531
300,0.050500,0.052974
350,0.043900,0.047004
400,0.042400,0.044887
450,0.047600,0.043664
500,0.040000,0.042004


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=1250, training_loss=0.08336397166252137, metrics={'train_runtime': 3865.3977, 'train_samples_per_second': 3.104, 'train_steps_per_second': 0.776, 'total_flos': 3.0655341943209984e+16, 'train_loss': 0.08336397166252137, 'epoch': 1.25})

In [11]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

SAVE_PATH = f"llama2_bias_finetune_{SUBSET_SIZE}"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

model.push_to_hub(f"moosejuice13/{SAVE_PATH}")
tokenizer.push_to_hub(f"moosejuice13/{SAVE_PATH}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   0%|          | 50.8kB /  160MB,   ???B/s  
Processing Files (0 / 1)                :   1%|          | 1.17MB /  160MB, 5.87MB/s  
Processing Files (0 / 1)                :   4%|▎         | 5.65MB /  160MB, 14.4MB/s  
Processing Files (0 / 1)                :   9%|▉         | 15.2MB /  160MB, 25.4MB/s  
Processing Files (0 / 1)                :  15%|█▌        | 24.1MB /  160MB, 30.4MB/s  
Processing Files (0 / 1)                :  21%|██        | 33.1MB /  160MB, 33.3MB/s  
Processing Files (0 / 1)                :  26%|██▌       | 41.5MB /  160MB, 34.9MB/s  
Processing Files (0 / 1)                :  30%|███       | 48.7MB /  160MB, 34.9MB/s  
Processing Files (0 / 1)                :  33%|███▎      | 53.2MB /  160MB, 33.4MB/s  
Processing Files (0 / 1)                :  37%|███▋      | 59.4MB /  160MB, 33.2MB/s  
Processing Files (0 / 1)                :  41%

Saved model to https://huggingface.co/moosejuice13/llama2_bias_finetune_5000


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  500kB /  500kB,   ???B/s  

Processing Files (1 / 1)                : 100%|██████████|  500kB /  500kB,  0.00B/s  
New Data Upload                         : |          |  0.00B /  0.00B,  0.00B/s  
  ..._bias_finetune_5000\tokenizer.model: 100%|██████████|  500kB /  500kB            
